# Notebook 01: Data Pipeline

Downloads and processes raw data. Produces abagym_antibody.csv, abagym_sequences.csv, sabdab_affinity.csv on Drive. This notebook was already run successfully. It exists for documentation and reproducibility -- re-run to verify outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys

REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
REPO_DIR = '/content/antibody-property-prediction'
BRANCH = 'implementation'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project/Antibody_Project')
# DATA_DIR is in the repo (data/ at repo root) -- comes from src.config
EMBEDDING_DIR = DRIVE_ROOT / 'embeddings'
RESULTS_DIR = DRIVE_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set.")

In [ ]:
!apt-get install -y hmmer
!pip install -q fair-esm ablang2 anarci wandb

In [ ]:
!pip install -q --upgrade ipython

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
subprocess.run(['find', '/content/antibody-property-prediction', '-type', 'd',
                '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
               capture_output=True)
print("Autoreload enabled, pycache cleared.")

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import wandb
wandb.login()

## Imports

In [1]:
import re
import json
import subprocess

import pandas as pd
from pathlib import Path

from src.config import DATA_DIR, DRIVE_ROOT, ABAGYM_DATASETS, N_MUTATIONS, N_SABDAB
from src.data.abagym import (
    load_abagym_antibody,
    load_abagym_sequences,
    PDB_FILE_STEMS,
    extract_chain_residues,
    build_anarci_mapping,
    reconstruct_mutant_sequence,
)
from src.data.sabdab import load_sabdab, parse_sabdab_raw, SABDAB_URL

## AbAgym Download

Download DMS datasets from github.com/3BioCompBio/AbAgym. One CSV per antibody.

In [4]:
# Clone the AbAgym repo to Drive (cached on re-run)
ABAGYM_CLONE_DIR = DRIVE_ROOT / 'AbAgym_repo'

if not ABAGYM_CLONE_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/3BioCompBio/AbAgym.git',
         str(ABAGYM_CLONE_DIR)],
        check=True,
    )
    print(f"Cloned to {ABAGYM_CLONE_DIR}")
else:
    print(f"Using cached clone at {ABAGYM_CLONE_DIR}")

print("\nRepo contents:")
for p in sorted(ABAGYM_CLONE_DIR.iterdir()):
    print(f"  {p.name}")

Using cached clone at /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/AbAgym_repo

Repo contents:
  .git
  AbAgym_data_full.csv.zip
  AbAgym_data_full_interface.csv
  AbAgym_data_non-redundant.csv.zip
  AbAgym_data_non-redundant_interface.csv
  AbAgym_metadata.csv
  DMS_big_table_PDB_files
  PDB_files.zip
  README.md


### AbAgym repo file descriptions

| File | Rows | Notes |
|---|---|---|
| `AbAgym_data_full.csv.zip` | 572,719 | All 68 DMS experiments, all mutations. Columns named `chain`, `mut_name` (singular) |
| `AbAgym_data_non-redundant.csv.zip` | 323,752 | Redundant experiments removed at dataset level. Columns already named `chains`, `mut_names` (our schema) |
| `AbAgym_data_full_interface.csv` | 37,259 | Full dataset filtered to interface-adjacent residues only |
| `AbAgym_data_non-redundant_interface.csv` | 36,541 | Non-redundant filtered to interface residues only |
| `AbAgym_metadata.csv` | 68 | One row per experiment with antigen, PDB ID, publication DOI |

Our 5 datasets have the same 5318 rows in both zip files. We use the non-redundant CSV since its column names already match our schema.

In [5]:
import zipfile

# Extract PDB files -- zip extracts to DMS_big_table_PDB_files/
pdb_dir = ABAGYM_CLONE_DIR / 'DMS_big_table_PDB_files'
if not pdb_dir.exists():
    with zipfile.ZipFile(ABAGYM_CLONE_DIR / 'PDB_files.zip') as zf:
        zf.extractall(ABAGYM_CLONE_DIR)
    print("Extracted PDB_files.zip")
else:
    print("PDB files already extracted")

print(f"\nChecking our 5 PDB files in {pdb_dir.name}/:")
for dms_name, stem in PDB_FILE_STEMS.items():
    pdb_path = pdb_dir / f'{stem}.pdb'
    status = "OK" if pdb_path.exists() else "MISSING"
    print(f"  [{status}] {stem}.pdb")

PDB files already extracted

Checking our 5 PDB files in DMS_big_table_PDB_files/:
  [OK] G6_27_30A_corrected_4zfg.pdb
  [OK] Cetuximab_1yy9.pdb
  [OK] D441_1mlc.pdb
  [OK] G6_27_30A_corrected_4zff.pdb
  [OK] trastuzumab_8pwh.pdb


In [6]:
# Extract and inspect the non-redundant DMS CSV
data_zip = ABAGYM_CLONE_DIR / 'AbAgym_data_non-redundant.csv.zip'
with zipfile.ZipFile(data_zip) as zf:
    csv_name = zf.namelist()[0]
    data_csv_path = ABAGYM_CLONE_DIR / csv_name
    if not data_csv_path.exists():
        zf.extractall(ABAGYM_CLONE_DIR)
        print(f"Extracted {csv_name}")
    else:
        print(f"Using cached {csv_name}")

raw_df = pd.read_csv(data_csv_path, dtype={'site': str})
print(f"Shape: {raw_df.shape}")
print(f"Columns: {raw_df.columns.tolist()}")
print(f"\nAll DMS_names ({raw_df['DMS_name'].nunique()} total):")
print(sorted(raw_df['DMS_name'].unique()))
print()
print(raw_df.head(3))

Extracted AbAgym_data_non-redundant.csv
Shape: (323752, 11)
Columns: ['DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation', 'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score', 'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance']

All DMS_names (68 total):
['Ang2_2017_G6', 'COVID-19_2021a_AZD1061', 'COVID-19_2021a_AZD8895', 'COVID-19_2021c_C002', 'COVID-19_2021c_C105', 'COVID-19_2021c_C110', 'COVID-19_2021c_C135', 'COVID-19_2021c_C144', 'COVID-19_2021c_CR3022', 'COVID-19_2021c_LY-CoV016', 'COVID-19_2021c_LY-CoV555', 'COVID-19_2021c_REGN10933', 'COVID-19_2021c_REGN10987', 'COVID-19_2021d_S2D106', 'COVID-19_2021d_S2E12', 'COVID-19_2021d_S2H13', 'COVID-19_2021d_S2H14', 'COVID-19_2021d_S2H97', 'COVID-19_2021d_S2X259', 'COVID-19_2021d_S2X35', 'COVID-19_2021d_S304', 'COVID-19_2021d_S309', 'COVID-19_2022_BD55-5840', 'COVID-19_2022_C119', 'COVID-19_2022_C121', 'COVID-19_2022_COV2-2130', 'COVID-19_2022_COV2-2196', 'COVID-19_2022_COVA2-04', 'COVID-19_2022_LY-C

In [7]:
# Filter to our 5 datasets -- columns already match our schema
combined_df = raw_df[raw_df['DMS_name'].isin(ABAGYM_DATASETS)].copy().reset_index(drop=True)

print(f"Rows: {len(combined_df)} (expected {N_MUTATIONS})")
print(f"Datasets present: {sorted(combined_df['DMS_name'].unique())}")
print(f"Columns: {combined_df.columns.tolist()}")
print()
print(combined_df[['DMS_name', 'chains', 'site', 'wildtype', 'mutation', 'mut_names']].head())

Rows: 5318 (expected 5318)
Datasets present: ['Ang2_2017_G6', 'EGFR_2013_Cetuximab', 'HER2_2021_trastuzumab', 'VEGF_2017b_G6', 'lysozyme_2019_D441']
Columns: ['DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation', 'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score', 'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance']

       DMS_name chains site wildtype mutation mut_names
0  Ang2_2017_G6      H  100        P        A    PH100A
1  Ang2_2017_G6      H  100        P        C    PH100C
2  Ang2_2017_G6      H  100        P        D    PH100D
3  Ang2_2017_G6      H  100        P        E    PH100E
4  Ang2_2017_G6      H  100        P        F    PH100F


In [8]:
# Build ANARCI mappings and wildtype sequences for all 5 antibodies
wt_sequences    = {}   # dms_name -> {'H': str, 'L': str}
anarci_mappings = {}   # dms_name -> {'H': mapping_dict, 'L': mapping_dict}

for dms_name in ABAGYM_DATASETS:
    stem     = PDB_FILE_STEMS[dms_name]
    pdb_path = pdb_dir / f'{stem}.pdb'

    h_residues = extract_chain_residues(str(pdb_path), 'H')
    l_residues = extract_chain_residues(str(pdb_path), 'L')

    h_map, _, _ = build_anarci_mapping(dms_name, 'H', h_residues)
    l_map, _, _ = build_anarci_mapping(dms_name, 'L', l_residues)

    wt_sequences[dms_name]    = {
        'H': ''.join(aa for _, aa in h_residues),
        'L': ''.join(aa for _, aa in l_residues),
    }
    anarci_mappings[dms_name] = {'H': h_map, 'L': l_map}
    print(f"  {dms_name}: H={len(h_residues)} res, L={len(l_residues)} res")

print("\nDone. Spot-check Ang2_2017_G6 chain H site 100A:")
sample = anarci_mappings['Ang2_2017_G6']['H'].get('100A')
print(f"  {sample}")  # expect imgt_pos=113, region=CDR_H3

  Ang2_2017_G6: H=215 res, L=213 res
  EGFR_2013_Cetuximab: H=220 res, L=211 res
  lysozyme_2019_D441: H=218 res, L=214 res
  VEGF_2017b_G6: H=211 res, L=213 res
  HER2_2021_trastuzumab: H=220 res, L=214 res

Done. Spot-check Ang2_2017_G6 chain H site 100A:
  {'imgt_pos': 113, 'imgt_ins': '', 'region': 'CDR_H3', 'seq_idx': 104}


## SAbDab Download

Download binding affinity data from Zenodo DOI 10.5281/zenodo.13120765 (Apache 2.0). Do NOT use PyTDC -- incompatible with Colab numpy.

In [9]:
import urllib.request

raw_sabdab_path = DRIVE_ROOT / 'sabdab_raw.csv'

if not raw_sabdab_path.exists():
    urllib.request.urlretrieve(SABDAB_URL, str(raw_sabdab_path))
    print(f"Downloaded SAbDab to {raw_sabdab_path}")
else:
    print(f"Using cached download at {raw_sabdab_path}")

raw_sabdab_df = pd.read_csv(raw_sabdab_path)
print(f"Raw shape: {raw_sabdab_df.shape}")
print(f"Columns: {raw_sabdab_df.columns.tolist()}")

sabdab_df = parse_sabdab_raw(raw_sabdab_df)
print(f"\nCleaned: {len(sabdab_df)} rows (expected {N_SABDAB})")
print(sabdab_df[['Antibody_ID', 'pKd']].describe())

Downloaded SAbDab to /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/sabdab_raw.csv
Raw shape: (493, 5)
Columns: ['Antibody_ID', 'Antibody', 'Antigen_ID', 'Antigen', 'Y']

Cleaned: 491 rows (expected 491)
              pKd
count  491.000000
mean     8.228588
std      1.504643
min      3.698970
25%      7.253104
50%      8.204815
75%      9.148785
max     12.397940


## Mutant Sequence Reconstruction

Reconstruct full mutant sequences from wildtype + single-point substitution. Store in mutant_heavy_seq and mutant_light_seq columns.

In [10]:
mutant_heavy_seqs = []
mutant_light_seqs = []

for row in combined_df.itertuples(index=False):
    h_wt    = wt_sequences[row.DMS_name]['H']
    l_wt    = wt_sequences[row.DMS_name]['L']
    mapping = anarci_mappings[row.DMS_name]

    mut_h, mut_l = reconstruct_mutant_sequence(
        heavy_seq=h_wt,
        light_seq=l_wt,
        chain=row.chains,
        site=str(row.site),
        wildtype_aa=row.wildtype,
        mutant_aa=row.mutation,
        mapping=mapping,
    )
    mutant_heavy_seqs.append(mut_h)
    mutant_light_seqs.append(mut_l)

combined_df = combined_df.copy()
combined_df['mutant_heavy_seq'] = mutant_heavy_seqs
combined_df['mutant_light_seq'] = mutant_light_seqs

print(f"Mutant sequences added. Shape: {combined_df.shape}")

# Sanity: each mutant seq should differ from wildtype at exactly 1 position
sample = combined_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    chain   = row['chains']
    wt_seq  = wt_sequences[row['DMS_name']][chain]
    mut_seq = row['mutant_heavy_seq'] if chain == 'H' else row['mutant_light_seq']
    diffs   = sum(a != b for a, b in zip(wt_seq, mut_seq))
    print(f"  {row['DMS_name']} {chain}:{row['wildtype']}{row['site']}{row['mutation']} -> {diffs} diff(s)")

Mutant sequences added. Shape: (5318, 13)
  lysozyme_2019_D441 H:F64E -> 1 diff(s)
  lysozyme_2019_D441 L:C88V -> 1 diff(s)
  lysozyme_2019_D441 L:I29T -> 1 diff(s)
  EGFR_2013_Cetuximab L:I55K -> 1 diff(s)
  Ang2_2017_G6 H:G54K -> 1 diff(s)


## CDR/FR Mapping

Map PDB position labels to IMGT positions using ANARCI. Requires HMMER (installed in cell 3). Pipeline: PDB position -> ANARCI alignment -> IMGT position -> CDR/FR region label.

Verification: 4zfg chain H, PDB 100A -> IMGT 113 -> CDR_H3.

In [11]:
regions = []
for row in combined_df.itertuples(index=False):
    chain_map = anarci_mappings[row.DMS_name][row.chains]
    site = str(row.site)
    assert site in chain_map, (
        f"Site {site!r} not found in {row.DMS_name} chain {row.chains} mapping"
    )
    regions.append(chain_map[site]['region'])

combined_df = combined_df.copy()
combined_df['region'] = regions

print("Region distribution:")
print(combined_df['region'].value_counts())
print(f"\nUnique regions: {sorted(combined_df['region'].unique())}")

Region distribution:
region
FR        2188
CDR_H3     851
CDR_L3     646
CDR_H2     558
CDR_L1     433
CDR_H1     415
CDR_L2     227
Name: count, dtype: int64

Unique regions: ['CDR_H1', 'CDR_H2', 'CDR_H3', 'CDR_L1', 'CDR_L2', 'CDR_L3', 'FR']


## Verification

Assert expected row counts and column presence before saving.

In [12]:
expected_cols = [
    'DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation',
    'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score',
    'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance',
    'mutant_heavy_seq', 'mutant_light_seq', 'region',
]

assert len(combined_df) == N_MUTATIONS, (
    f"Expected {N_MUTATIONS} rows, got {len(combined_df)}"
)
missing = [c for c in expected_cols if c not in combined_df.columns]
assert not missing, f"Missing columns: {missing}"

assert len(sabdab_df) == N_SABDAB, (
    f"Expected {N_SABDAB} rows, got {len(sabdab_df)}"
)

print(f"combined_df: {combined_df.shape}  (expected ({N_MUTATIONS}, {len(expected_cols)}))")
print(f"sabdab_df:   {sabdab_df.shape}    (expected ({N_SABDAB}, 7))")
print("All assertions passed.")

combined_df: (5318, 14)  (expected (5318, 14))
sabdab_df:   (491, 7)    (expected (491, 7))
All assertions passed.


## Save to Drive

In [13]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

# abagym_antibody.csv -- 5318 mutation rows, 14 columns
antibody_cols = [
    'DMS_name', 'PDB_file', 'chains', 'site', 'wildtype', 'mutation',
    'mut_names', 'DMS_score', 'MinMax_normalized_DMS_score',
    'Rank_quartile_normalized_DMS_score', 'closest_interface_atom_distance',
    'mutant_heavy_seq', 'mutant_light_seq', 'region',
]
combined_df[antibody_cols].to_csv(DATA_DIR / 'abagym_antibody.csv', index=False)

# abagym_sequences.csv -- 5 rows (one per antibody), ANARCI mappings as JSON strings
sequences_rows = []
for dms_name in ABAGYM_DATASETS:
    sequences_rows.append({
        'dms_name':  dms_name,
        'heavy_seq': wt_sequences[dms_name]['H'],
        'light_seq': wt_sequences[dms_name]['L'],
        'mapping_H': json.dumps(anarci_mappings[dms_name]['H']),
        'mapping_L': json.dumps(anarci_mappings[dms_name]['L']),
    })
sequences_df = pd.DataFrame(sequences_rows)
sequences_df.to_csv(DATA_DIR / 'abagym_sequences.csv', index=False)

# sabdab_affinity.csv -- 491 rows
sabdab_df.to_csv(DATA_DIR / 'sabdab_affinity.csv', index=False)

print(f"Saved to {DATA_DIR}:")
print(f"  abagym_antibody.csv   {len(combined_df)} rows x {len(antibody_cols)} cols")
print(f"  abagym_sequences.csv  {len(sequences_df)} rows x {len(sequences_df.columns)} cols")
print(f"  sabdab_affinity.csv   {len(sabdab_df)} rows x {len(sabdab_df.columns)} cols")

Saved to /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project/data:
  abagym_antibody.csv   5318 rows x 14 cols
  abagym_sequences.csv  5 rows x 5 cols
  sabdab_affinity.csv   491 rows x 7 cols
